In [24]:
import pandas as pd
import boto3
import io
import gzip
import json

In [4]:
# Begin demo

In [16]:
# Create Dataframe

s3_client = boto3.client("s3")
bucket = "s3-3601-final-project"
prefix = "tomtom/traffic/"

response = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
df_list = []

if "Contents" in response:
    for obj in response["Contents"]:
        key = obj["Key"]

        if not key.endswith(".json.gz"):
            continue

        file_obj = s3_client.get_object(Bucket=bucket, Key=key)
        raw_bytes = file_obj["Body"].read()
        try:
            data_bytes = gzip.decompress(raw_bytes)
        except gzip.BadGzipFile:
            data_bytes = raw_bytes

        obj_df = pd.read_csv(io.BytesIO(data_bytes))
        df_list.append(obj_df)
        
        final_df = pd.concat(df_list, ignore_index=True)

    if df_list:
        final_df = pd.concat(df_list, ignore_index=True)
        final_df.head()
final_df.head(20)

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month,speed_ratio,holiday
0,2026-09-26 19:03:46,35.307778,-80.724583,19,27,NaN,1,5,19,9,0.703704,Not a federal holiday
1,2026-09-26 19:03:46,35.313611,-80.701944,40,40,NaN,1,5,19,9,1.000000,Not a federal holiday
2,2026-09-26 19:03:46,35.311361,-80.711222,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday
3,2026-09-26 19:03:47,35.309306,-80.718056,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday
4,2026-09-26 19:03:47,35.306917,-80.722639,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday
5,2026-09-26 19:03:47,35.293111,-80.749361,17,28,NaN,1,5,19,9,0.607143,Not a federal holiday
6,2026-09-26 19:03:47,35.294889,-80.742111,55,55,NaN,1,5,19,9,1.000000,Not a federal holiday
7,2026-09-26 19:03:47,35.299639,-80.745750,19,28,NaN,1,5,19,9,0.678571,Not a federal holiday
8,2026-09-26 19:03:47,35.287972,-80.759472,20,25,NaN,1,5,19,9,0.800000,Not a federal holiday
9,2026-09-26 19:03:47,35.301861,-80.752333,18,26,NaN,1,5,19,9,0.692308,Not a federal holiday


In [27]:
response2 = s3_client.get_object(Bucket=bucket, Key="ncdot/traffic/traffic_counts.csv")


raw_bytes = response2["Body"].read()
try:
    data_bytes = gzip.decompress(raw_bytes)
except gzip.BadGzipFile:
    data_bytes = raw_bytes

df2_final = pd.read_csv(io.BytesIO(data_bytes))


df2_final.head(20)

,attributes.LocationID,attributes.COUNTY,attributes.ROUTE,attributes.LOCATION,attributes.AADT_2022,longitude,latitude
0,600000044,MECKLENBURG,US 29/NC 49,EAST OF STETSON DR,29500,-80.759574,35.287985
1,600000063,MECKLENBURG,NC 49,WEST OF I-485,32500,-80.711385,35.311292
2,600000065,MECKLENBURG,US 29,SOUTH OF UNIVERSITY POINTE BLVD,18000,-80.755360,35.294440
3,600000107,MECKLENBURG,NC 49,EAST OF SUTHER RD,26500,-80.727801,35.303270
4,600000108,MECKLENBURG,NC 49,WEST OF SUTHER RD,27500,-80.729977,35.301876
5,600000145,MECKLENBURG,NC 49,EAST OF SR 5394 MALLARD CK CH RD,42000,-80.717633,35.309346
6,600000412,MECKLENBURG,NC 24 (WT HARRIS BLVD),SOUTH OF US 29,47500,-80.748333,35.304176
7,600000414,MECKLENBURG,NC 24 (E WT HARRIS BLVD),EAST OF SR 5316,51500,-80.739913,35.293191
8,600000829,MECKLENBURG,NC 49,EAST OF SR 2939,32500,-80.722975,35.306539
9,600001182,MECKLENBURG,SR 2833,NORTH OF NC 49,15500,-80.724601,35.306768


In [32]:
# Combine the two dataframes

df2_final["longitude"]=df2_final["longitude"].astype(str).str.strip()
final_df["longitude"]=final_df["longitude"].astype(str).str.strip()

location_map = df2_final.set_index("longitude")["attributes.AADT_2022"]

final_df["AADT_2022"] = final_df["longitude"].map(location_map)

final_df.head()

,timestamp,latitude,longitude,current_speed,free_flow_speed,current_delay,confidence,day_of_week,hour,month,speed_ratio,holiday,AADT_20220,AADT_2022
0,2026-09-26 19:03:46,35.307778,-80.724583,19,27,NaN,1,5,19,9,0.703704,Not a federal holiday,NaN,NaN
1,2026-09-26 19:03:46,35.313611,-80.701944,40,40,NaN,1,5,19,9,1.000000,Not a federal holiday,NaN,NaN
2,2026-09-26 19:03:46,35.311361,-80.711222,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday,NaN,NaN
3,2026-09-26 19:03:47,35.309306,-80.718056,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday,NaN,NaN
4,2026-09-26 19:03:47,35.306917,-80.722639,26,31,NaN,1,5,19,9,0.838710,Not a federal holiday,NaN,NaN
